In [2]:
# Cell 1: Imports and Global Parameters

import os
import subprocess
import pandas as pd

# — USER PARAMETERS — 

# 1) Input FASTA (amino-acid sequences containing full-length heavy chain or heavy+light)
import pandas as pd
import numpy as np
import openpyxl
import time




In [19]:
date_str = time.strftime("%Y%m%d")
date_str

dataset = pd.read_excel('/Users/ahleyliu/Documents/Active_Projects/Antibody_database/RSV_database/RSV_yixing0501.xlsx')
#Extract seq
dataset_selected = dataset[['name','VH_nuc','VL_nuc','age']].copy()
dataset_selected = dataset_selected[dataset_selected['VH_nuc'].notna()]
#dataset_selected = dataset_selected[dataset_selected['VL_nuc'].notna()]
dataset_selected.columns = ['Name','Heavy_nuc','Light_nuc','type']
final_df = pd.concat([dataset_selected],ignore_index=True)
final_df['Name'] = final_df['Name'].astype(str)
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
from Bio import SeqIO
import os

os.makedirs("../temp", exist_ok=True)

heavy_records = [
    SeqRecord(
        Seq(str(row['Heavy_nuc'])),
        id=f"{row['type']}_{row['Name']}",
        description=""
    )
    for _, row in final_df.iterrows()
    if pd.notnull(row['Heavy_nuc']) and row['Heavy_nuc'] != ""
]

SeqIO.write(heavy_records, "../temp/rsvH.fasta", "fasta")
print(f"Heavy chain sequences written: {len(heavy_records)}")


light_records = [
    SeqRecord(
        Seq(str(row['Light_nuc'])),
        id=f"{row['type']}_{row['Name']}",
        description=""
    )
    for _, row in final_df.iterrows()
    if pd.notnull(row['Light_nuc']) and row['Light_nuc'] != ""
]

SeqIO.write(light_records, "../temp/rsvL.fasta", "fasta")
print(f"Light chain sequences written: {len(light_records)}")



from collections import defaultdict

def unique_id(input_file,output_file):
    id_count = defaultdict(int)
    records = []
    
    for record in SeqIO.parse(input_file, "fasta"):
        id_count[record.id] += 1
        if id_count[record.id] > 1:
            record.id = f"{record.id}_{id_count[record.id]}"
            record.description = ""
        records.append(record)
    
    SeqIO.write(records, output_file, "fasta")

unique_id('../temp/rsvH.fasta','../temp/rsvH.dedup.fasta')
unique_id('../temp/rsvL.fasta','../temp/rsvL.dedup.fasta')


Heavy chain sequences written: 1927
Light chain sequences written: 1135


In [21]:
input_fasta = '../temp/rsvH.dedup.fasta'

In [1]:
import os
os.chdir('/Users/ahleyliu/Documents/Active_Projects/Antibody_database/changeo')
print(os.getcwd())  # 确认已经切换

/Users/ahleyliu/Documents/Active_Projects/Antibody_database/changeo


In [3]:
# ##这一步在跑igblast,会很慢
!python ./AssignGenes.py igblast \
    -s ./input/141415_0.fasta -b igblast -o ./temp/Tar_igblast.fmt7


#这一步速度还行,在归纳igblast结果
!python ./MakeDb.py igblast  \
    -s ./input/141415_0.fasta -r igblast/human/vdj \
    -i ./temp/Tar_igblast.fmt7 -o ./temp/Tar_igblast_db-pass.tsv

#这一步也超级慢，进行一个基于CDR3和gene_usage的聚类
!python ./DefineClones.py   \
    -d ./temp/Tar_igblast_db-pass.tsv --mode gene --act set\
    --model hh_s5f --dist 0.2  \
    --sf JUNCTION --norm len \
    -o ./temp/Tar_igblast_db-pass_clone-pass.tsv

#这一步给每个类计算一个germline,速度很快
!python ./CreateGermlines.py   \
    -d ./temp/Tar_igblast_db-pass_clone-pass.tsv  -r igblast/human/vdj    \
    -g dmask --cloned  \
    -o ./temp/Tar_igblast_db-pass_clone-pass_germ-pass.tsv

!python ./CreateGermlines.py   \
    -d ./temp/Tar_igblast_db-pass_clone-pass.tsv  -r igblast/human/vdj    \
    -g dmask --cloned  \
    -o ./output/result.tsv

/Users/ahleyliu/Documents/Active_Projects/Antibody_database/changeo/./AssignGenes.py:14: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import parse_version
WARNING> Output file ./temp/Tar_igblast.fmt7 already exists and will be overwritten.
usage: AssignGenes.py [--version] [-h]  ...
AssignGenes.py: error: igblastn executable not found
WARNING> Output file ./temp/Tar_igblast_db-pass.tsv already exists and will be overwritten.
         START> MakeDB
       COMMAND> igblast
  ALIGNER_FILE> Tar_igblast.fmt7
      SEQ_FILE> 141415_0.fasta
       ASIS_ID> False
    ASIS_CALLS> False
      VALIDATE> strict
      EXTENDED> False
INFER_JUNCTION> False

PROGRESS> 22:07:31 |Done                | 0.0 min

PROGRESS> 22:07:31 |                    |   0% (     0) 0.0 minTraceback (most recent call last):
  File "/Users/ahleyliu/Documents/Active_Projects/Antibody_database/changeo/./MakeDb.py", line 1112, in 

In [ ]:
Data=pd.read_csv("./output/result.tsv",sep="\t")
Data.to_csv("./output/result.csv",sep=",")
Data.iloc[0,:]

Data.iloc[0,:]

Hcloned_data = pd.read_csv(f'../results/rsv{date_str}_H_pass-clone_germ-pass.tsv',sep='\t')
Lcloned_data = pd.read_csv(f'../results/rsv{date_str}_L_pass-clone_germ-pass.tsv',sep='\t')

Hdict_type = dict(zip(final_df['Heavy_nuc'],final_df['type']))
Hcloned_data['type'] = Hcloned_data['sequence'].map(Hdict_type)
Hcloned_data['clone_size'] = Hcloned_data['clone_id'].map(Hcloned_data['clone_id'].value_counts())

Ldict_type = dict(zip(final_df['Light_nuc'],final_df['type']))
Lcloned_data['type'] = Lcloned_data['sequence'].map(Ldict_type)
Lcloned_data['clone_size'] = Lcloned_data['clone_id'].map(Lcloned_data['clone_id'].value_counts())

In [ ]:
#Clone Size
import matplotlib.pyplot as plt
import seaborn as sns

def clone_size_plot(cloned_data):
    clone_counts = cloned_data['clone_id'].value_counts()
    freq_distribution = clone_counts.value_counts()
    freq_distribution_grouped = freq_distribution.copy()
    freq_distribution_grouped.index = freq_distribution_grouped.index.map(lambda x: '10+' if x >= 10 else str(x))
    freq_distribution_grouped = freq_distribution_grouped.groupby(freq_distribution_grouped.index).sum()
    freq_distribution_grouped = freq_distribution_grouped.reindex(
        [str(i) for i in range(1, 10)] + ['10+']
    ).fillna(0) 
    
    plt.figure(figsize=(4, 3))
    sns.barplot(x=freq_distribution_grouped.index, y=freq_distribution_grouped.values, color='steelblue')
    plt.xlabel("Clone Size")
    plt.ylabel("Frequency")
    plt.title("Clone Size Plot")
    plt.tight_layout()
    plt.show()